# Implementación técnica para el acompañamiento académico en matemáticas
**Implementación técnica y preparación de entrega EP5 · versión 3**

Integrantes: Genesis Baeza, Jimena Galicia, Alejandra González, Constanza González y Diego Villar.

Asignatura: Gestión de Proyectos de Datos. 

Profesor: Ronald Villalobos Arrano. Sección 002D.

Este cuaderno desarrolla las ocho tareas de la guía y la reflexión grupal. El objetivo es estimar una señal de bajo rendimiento en **matemáticas** para apoyar la revisión docente.

Se usa exclusivamente la **versión 1 de Kaggle** de `neuralsorcerer/student-performance`. La fuente declara que los datos son **sintéticos**. Los resultados no demuestran eficacia en estudiantes reales, equidad institucional ni impacto de una intervención. El piloto de aproximadamente 1.500 estudiantes descrito en los informes es un contexto propuesto; estos millones de registros no pertenecen a ese colegio.

**Definición provisional:** clase 1 si `TestScore_Math < 60`, clase 0 en otro caso. El corte 60/100 permite desarrollar el ejercicio, pero no está fijado en las evaluaciones ni equivale automáticamente a reprobación en Chile. Debe ser validado por el docente/UTP. Se fija antes de entrenar y no se elige por los resultados de test.

**Límite temporal:** no hay fechas de medición ni resultados futuros identificados. Se excluyen las notas contemporáneas de las entradas, pero no se puede demostrar que asistencia o hábitos se midieron antes de matemáticas. Se trata de un prototipo de asociación y clasificación; para llamarlo alerta temprana validada se requieren datos longitudinales.

El modo predeterminado utiliza muestras uniformes reproducibles de **300.000 filas de train, 150.000 de validación y 150.000 de test**, tomadas recorriendo todos los archivos. No utiliza simplemente las primeras filas. Los conteos y controles estructurales recorren los archivos completos; entrenamiento, gráficos y métricas usan las muestras indicadas. Aumentar estos tamaños implica ejecutar nuevamente todo el flujo y no seleccionar el tamaño mirando resultados de test.

### Cómo se incorporan las evaluaciones previas
| Documento o requisito | Aplicación en el cuaderno |
|---|---|
| Evaluación 1 y RF-01 | Clasificación de bajo rendimiento en matemáticas y revisión humana |
| RF-03 | Auditoría de errores por género, etnia y cuartil socioeconómico, exportada por separado |
| RF-02 | CSV con probabilidades y niveles para una futura conexión a Power BI; no se entrega un dashboard terminado |
| RNF-05 y H3 | Semilla, versión de datos, huellas, versiones de software, pipeline y ficha de modelo |
| CRISP-DM | Comprensión del problema, datos, preparación, modelado, evaluación y prototipo de despliegue |
| Automatización por lotes | Inferencia y exportación con bitácora; sin envío automático de mensajes |
| H4 y RNF-02 | La aprobación ética, cifrado y control de accesos de producción siguen pendientes; este notebook no acredita su implementación |

**Corrección del diccionario:** Kaggle define `StudyHours` como horas de estudio **por día** (0–4). La evaluación 1 las describe como semanales. Aquí se utiliza la definición de la fuente, sin multiplicar ni dividir los valores originales.

## Cómo ejecutar esta versión
1. Guardar el notebook y reiniciar el kernel. Ejecutar todas las celdas en orden.
2. La descarga descomprimida ocupa aproximadamente 1,66 GB más temporales. La lectura por bloques reduce RAM, pero conserva hashes de todos los registros para los controles; no es de memoria constante.
3. Revisar el modelo elegido y el resumen automático. No forzar un ganador ni ajustar mirando test.
4. Al terminar, abrir la carpeta que imprime el último paso. Cada ejecución utiliza una carpeta nueva.
5. Usar el CSV de **dashboard/predicciones_actuales.csv** para Power BI. Conservar el ZIP de evidencia generado.

Las salidas anteriores se retiraron para no presentar resultados antiguos como una nueva ejecución. En el archivo recibido había una contradicción: las salidas seleccionaban HistGradientBoosting y el texto describía regresión logística. Esta versión genera la interpretación desde los resultados vigentes. El entrenamiento completo debe ejecutarse en su entorno; la revisión de esta entrega verifica estructura y el flujo de inferencia con datos de prueba, no repite la descarga y el experimento completo.


In [ ]:
import importlib.util, subprocess, sys
for modulo, paquete in [('numpy','numpy'),('pandas','pandas'),('matplotlib','matplotlib'),('sklearn','scikit-learn'),('joblib','joblib')]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',paquete])
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, hashlib, platform, time, gc
from datetime import datetime, timezone
import sklearn, joblib
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance
SEED=42
CORTE_MATH=60.0
HANDLE='neuralsorcerer/student-performance/versions/1'
TAMANOS={'train':300_000,'validation':150_000,'test':150_000}
from uuid import uuid4
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'_'+uuid4().hex[:8]
OUT=Path('salidas_matematicas')/RUN_ID; OUT.mkdir(parents=True, exist_ok=True)
versiones={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__, 'joblib':joblib.__version__}
print('Versiones:',versiones)
print('Configuración:',HANDLE,TAMANOS,'Corte matemáticas:',CORTE_MATH)

## 1 Cargar el dataset del proyecto
Los archivos se descargan con `dataset_download`, usando un identificador de versión fijo. Así se puede leer cada CSV con `pandas.read_csv(..., chunksize=...)`, sin construir un DataFrame de ocho millones de filas.


In [ ]:
ESPERADOS={'train':1_328_598_306,'validation':165_931_109,'test':166_057_940}
local=Path('datos_kaggle')
if all((local/f'{s}.csv').exists() for s in TAMANOS):
    base=local
    print('Se usan los CSV locales de datos_kaggle/.')
else:
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','kagglehub'])
    import kagglehub
    base=Path(kagglehub.dataset_download(HANDLE))
    print('Descarga de la versión 1 completada.')
rutas={s:base/f'{s}.csv' for s in TAMANOS}
COLUMNAS=['Age','Grade','Gender','Race','SES_Quartile','ParentalEducation','SchoolType','Locale',
 'TestScore_Math','TestScore_Reading','TestScore_Science','GPA','AttendanceRate','StudyHours',
 'InternetAccess','Extracurricular','PartTimeJob','ParentSupport','Romantic','FreeTime','GoOut']
for s,p in rutas.items():
    if p.stat().st_size != ESPERADOS[s]:
        raise ValueError(f'{s}: tamaño diferente al archivo original de la versión 1. Revisar procedencia; no mezclar versiones.')
    if list(pd.read_csv(p,nrows=0).columns) != COLUMNAS:
        raise ValueError(f'{s}: esquema incompatible con la versión usada en el proyecto.')
print('Los tres archivos coinciden con los tamaños y el esquema esperado de Kaggle v1.')

### Lectura por bloques y muestreo uniforme
A cada fila se le asigna una prioridad aleatoria con semilla fija. Se conservan las k menores entre todos los bloques: todas las filas del archivo tienen la misma probabilidad de inclusión. Los atributos de texto se leen como texto para que el hash de filas sea comparable entre archivos; los decimales mantienen precisión float64.

Se calculan conteos de nulos y huellas SHA-256 de los archivos completos para trazabilidad. Se conservan hashes de fila de 64 bits para buscar duplicados y coincidencias entre particiones: son un control práctico, no una identificación de estudiantes ni una prueba criptográfica de identidad. Si aparecen coincidencias, el proceso se detiene para revisarlas en vez de eliminarlas silenciosamente.

In [ ]:
def sha256_archivo(ruta):
    h=hashlib.sha256()
    with open(ruta,'rb') as f:
        for bloque in iter(lambda:f.read(8*1024*1024),b''): h.update(bloque)
    return h.hexdigest()
TEXTO={c:'string' for c in ['Gender','Race','ParentalEducation','SchoolType','Locale']}
DECIMALES={c:'float64' for c in ['TestScore_Math','TestScore_Reading','TestScore_Science','GPA','AttendanceRate','StudyHours']}
def leer_muestra(ruta,k,seed):
    rng=np.random.default_rng(seed); muestra=None; hashes=[]; total=0
    nulos=pd.Series(0,index=COLUMNAS,dtype='int64')
    for bloque in pd.read_csv(ruta,chunksize=100_000,dtype={**TEXTO,**DECIMALES}):
        nulos+=bloque.isna().sum()
        hashes.append(pd.util.hash_pandas_object(bloque,index=False).to_numpy())
        bloque['_fila_fuente']=np.arange(total,total+len(bloque))+2
        total+=len(bloque)
        bloque['_prioridad']=rng.random(len(bloque))
        candidato=bloque if muestra is None else pd.concat([muestra,bloque],ignore_index=True)
        muestra=candidato.nsmallest(k,'_prioridad')
    h=np.concatenate(hashes); unicos=np.unique(h)
    reporte={'filas_originales':total,'filas_muestra':len(muestra),'nulos_originales':int(nulos.sum()),
             'hashes_repetidos':int(len(h)-len(unicos)),'sha256':sha256_archivo(ruta)}
    return muestra.drop(columns='_prioridad').reset_index(drop=True),unicos,reporte
muestras={}; hashes={}; calidad={}
for i,s in enumerate(TAMANOS):
    muestras[s],hashes[s],calidad[s]=leer_muestra(rutas[s],TAMANOS[s],SEED+i)
    print(s,calidad[s])
print('\nControl de coincidencias entre archivos completos')
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    n=len(np.intersect1d(hashes[a],hashes[b],assume_unique=True))
    print(a,b,n)
    if n: raise ValueError('Hay coincidencias de hash entre particiones: revisar registros antes de entrenar.')
if any(v['hashes_repetidos'] for v in calidad.values()):
    raise ValueError('Revisar los posibles duplicados de origen antes de entrenar.')
del hashes; gc.collect()
pd.DataFrame(calidad).T.to_csv(OUT/'calidad_y_procedencia.csv')
print('\nPrimeras filas de la muestra train:\n',muestras['train'].head().to_string(index=False))

## 2 Analizar las clases y generar gráficos representativos
El análisis exploratorio se realiza en train. La clase 1 significa **puntaje de matemáticas inferior al corte provisional**, no diagnóstico de incapacidad ni certeza de fracaso futuro. La distribución de clases de validación y test se describe cuando se usa cada conjunto, sin redefinir el corte para equilibrarlos.

Las relaciones entre asistencia, estudio y resultado sirven para describir la muestra. No permiten concluir que aumentar una variable cause una mejora en el rendimiento.

In [ ]:
train=muestras['train']; valid=muestras['validation']; test=muestras['test']
def objetivo(datos):
    if datos.TestScore_Math.isna().any(): raise ValueError('No imputar la variable objetivo.')
    if not datos.TestScore_Math.between(0,100).all(): raise ValueError('Puntajes fuera de rango.')
    return (datos.TestScore_Math<CORTE_MATH).astype('int8')
y_train=objetivo(train)
conteo=y_train.value_counts().reindex([0,1],fill_value=0)
print(pd.DataFrame({'Clase':['Sin bajo puntaje','Bajo puntaje'],'Cantidad':conteo.values,'Porcentaje':conteo.values/len(train)*100}).round(2).to_string(index=False))
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].bar(['Sin bajo puntaje','Bajo puntaje'],conteo.values,color=['#287c8e','#dc7845'])
ax[0].set_title('Clases en train');ax[0].set_ylabel('Registros');ax[0].set_ylim(0,conteo.max()*1.2)
for i,n in enumerate(conteo):ax[0].text(i,n,f'{n:,}\n{n/len(train):.1%}',ha='center',va='bottom')
ax[1].hist(train.TestScore_Math,bins=40,color='#287c8e');ax[1].axvline(CORTE_MATH,color='#b54229',linestyle='--',label='Corte provisional 60')
ax[1].set_title('Puntaje de matemáticas');ax[1].set_xlabel('Puntos');ax[1].legend()
ax[2].boxplot([train.loc[y_train==c,'AttendanceRate'].to_numpy()*100 for c in [0,1]],showfliers=False)
ax[2].set_xticks([1,2],['Sin bajo puntaje','Bajo puntaje']);ax[2].set_ylabel('Asistencia (%)');ax[2].set_title('Asistencia por clase')
plt.tight_layout();plt.savefig(OUT/'clases_train.png',dpi=140);plt.show()
print('\nCorrelaciones descriptivas en train:\n',train[['TestScore_Math','AttendanceRate','StudyHours']].corr().round(3).to_string())

## 3 Preparar los datos y seleccionar variables
**Variable dependiente:** `RiesgoMatematicas = 1[TestScore_Math < 60]`.

| Variables | Decisión y motivo |
|---|---|
| AttendanceRate, StudyHours | Predictores de asistencia y hábitos, relevantes para acompañamiento; disponibilidad temporal por confirmar |
| Grade | Nivel escolar; Age se excluye por redundancia |
| InternetAccess, ParentSupport | Acceso y apoyo; posibles proxies que requieren auditoría |
| Extracurricular, PartTimeJob, FreeTime, GoOut | Contexto de actividades y tiempo; no deben usarse para responsabilizar al estudiante |
| TestScore_Math | Solo construye la clase; nunca entra al modelo |
| GPA, TestScore_Reading, TestScore_Science | Excluidas: no se conoce si contienen información posterior o del mismo periodo que el resultado |
| Gender, Race, SES_Quartile | Solo auditoría de equidad, separada de la predicción |
| Romantic, ParentalEducation, SchoolType, Locale | Excluidas del prototipo por minimización de datos |
| _fila_fuente | Solo trazabilidad técnica, no predictor ni identificador real de estudiante |

Las entradas seleccionadas son numéricas o binarias; no hace falta codificar texto. Se imputa con mediana aprendida **solo en train** dentro del pipeline. La regresión logística además estandariza con estadísticas de train. No se aplica sobremuestreo antes de separar conjuntos. No se confunde normalizar con dividir los datos.

La exclusión de atributos protegidos no garantiza equidad: asistencia o acceso a recursos pueden transportar información social. Por eso se conserva una auditoría separada por subgrupos.

In [ ]:
FEATURES=['Grade','AttendanceRate','StudyHours','InternetAccess','Extracurricular','PartTimeJob','ParentSupport','FreeTime','GoOut']
AUDITORIA=['Gender','Race','SES_Quartile']
LIMITES={'Grade':(9,12),'AttendanceRate':(0.70,1),'StudyHours':(0,4),'FreeTime':(1,5),'GoOut':(1,5)}
BINARIAS=['InternetAccess','Extracurricular','PartTimeJob','ParentSupport']
def validar_entradas(datos):
    faltan=set(FEATURES)-set(datos.columns)
    if faltan: raise ValueError(f'Faltan columnas: {sorted(faltan)}')
    x=datos[FEATURES].apply(pd.to_numeric,errors='raise')
    if x.empty or x.isna().all(axis=1).any(): raise ValueError('Lote vacío o fila sin datos predictivos.')
    if np.isinf(x.to_numpy()).any(): raise ValueError('Valores infinitos.')
    for c,(lo,hi) in LIMITES.items():
        if not x[c].dropna().between(lo,hi).all(): raise ValueError(f'{c} fuera de rango {lo}–{hi}.')
    for c in BINARIAS:
        if not x[c].dropna().isin([0,1]).all(): raise ValueError(f'{c} debe ser 0 o 1.')
    for c in ['Grade','FreeTime','GoOut']:
        if not (x[c].dropna()%1==0).all(): raise ValueError(f'{c} debe ser entero.')
    return x
X_train=validar_entradas(train)
assert not set(FEATURES)&set(AUDITORIA+['GPA','TestScore_Math','TestScore_Reading','TestScore_Science'])
print('Predictores:',FEATURES)
print('Faltantes en predictores de train:',int(X_train.isna().sum().sum()))
# AttendanceRate 0.70–1 y Grade 9–12 son límites del dominio sintético,
# no límites universales de asistencia o cursos de un colegio chileno.


## 4 Separar los datos de entrenamiento validación y test
Se respetan los archivos de origen: no se concatenan para volver a dividir. El muestreo se hizo independientemente dentro de cada uno, sin compartir filas. Train ajusta parámetros y transformaciones; validación selecciona modelo y umbral; test mide el resultado una vez cerrada la selección.

**Antecedente que debe declararse:** test ya fue explorado en la evaluación. Este nuevo experimento mantiene sus muestras fuera del entrenamiento, pero una validación externa futura aportaría una comprobación más fuerte. La versión actual no tiene identificadores de estudiante: no se puede auditar repetición de personas, solo coincidencias de registros.

In [ ]:
X_val=validar_entradas(valid); y_val=objetivo(valid)
print(pd.DataFrame([{'Conjunto':s,'Filas originales':calidad[s]['filas_originales'],'Filas utilizadas':len(muestras[s])} for s in TAMANOS]).to_string(index=False))
print('Positivos train:',int(y_train.sum()),'| validación:',int(y_val.sum()))
assert y_train.nunique()==2 and y_val.nunique()==2

## 5 Entrenar modelos y analizar si la preparación fue suficiente
Se comparan una referencia mayoritaria, regresión logística e HistGradientBoosting (árboles de boosting). La referencia permite comprobar que la exactitud por sí sola puede ser engañosa con pocos casos positivos.

**Regla de selección fijada antes de test:** para cada candidato, buscar el umbral de probabilidad que maximiza F2 en validación en una grilla de 0,02 a 0,80. Luego elegir el candidato con mayor F2 de validación. F2 da más peso al recall que a la precisión, apropiado como propuesta inicial para acompañamiento. El costo de falsas alertas y la capacidad de atención todavía deben acordarse con UTP.

El corte del puntaje (60 puntos) y el umbral de probabilidad cumplen funciones distintas: uno define la respuesta real del ejercicio; el otro decide cuándo emitir una señal del modelo. No se ajustan mirando test. Se presenta una comparación acotada, no una búsqueda exhaustiva de hiperparámetros.

In [ ]:
modelos={
 'Referencia mayoritaria':Pipeline([('imputar',SimpleImputer(strategy='median')),('modelo',DummyClassifier(strategy='most_frequent'))]),
 'Regresión logística':Pipeline([('imputar',SimpleImputer(strategy='median')),('escalar',StandardScaler()),('modelo',LogisticRegression(max_iter=500,random_state=SEED))]),
 'HistGradientBoosting':Pipeline([('imputar',SimpleImputer(strategy='median')),('modelo',HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=15,l2_regularization=1,early_stopping=False,random_state=SEED))])}
def medir(y,p,t):
    pr=(p>=t).astype(int)
    return {'Accuracy':accuracy_score(y,pr),'Precision':precision_score(y,pr,zero_division=0),
    'Recall':recall_score(y,pr,zero_division=0),'F1':f1_score(y,pr,zero_division=0),
    'F2':fbeta_score(y,pr,beta=2,zero_division=0),'ROC_AUC':roc_auc_score(y,p),'AP':average_precision_score(y,p)}
val_filas=[];umbrales={};probas_val={};ajustes=[]
for nombre,m in modelos.items():
    inicio=time.perf_counter();m.fit(X_train,y_train)
    p=m.predict_proba(X_val)[:,1];probas_val[nombre]=p
    ts=np.array([0.5]) if nombre=='Referencia mayoritaria' else np.round(np.arange(0.02,0.801,0.02),2)
    f2s=[fbeta_score(y_val,p>=t,beta=2,zero_division=0) for t in ts]
    t=float(ts[int(np.argmax(f2s))]);umbrales[nombre]=t
    for tt,ss in zip(ts,f2s):ajustes.append({'Modelo':nombre,'Umbral':tt,'F2_validacion':ss})
    val_filas.append({'Modelo':nombre,'Umbral':t,'Segundos':time.perf_counter()-inicio,**medir(y_val,p,t)})
val_metricas=pd.DataFrame(val_filas).set_index('Modelo')
elegido=val_metricas.drop(index='Referencia mayoritaria').F2.idxmax()
final=modelos[elegido];umbral=umbrales[elegido]
print(val_metricas.round(4).to_string())
print('\nElegido por validación:',elegido,'| umbral:',umbral)
f2_train=fbeta_score(y_train,final.predict_proba(X_train)[:,1]>=umbral,beta=2)
print('F2 train:',round(f2_train,4),'| F2 validación:',round(val_metricas.loc[elegido,'F2'],4))
val_metricas.to_csv(OUT/'metricas_validacion.csv');pd.DataFrame(ajustes).to_csv(OUT/'busqueda_umbrales.csv',index=False)

### Revisión de la preparación y explicación de variables
El pipeline entrena sin la variable que revela la respuesta y sin atributos protegidos como entradas. La comparación de F2 entre train y validación permite observar una posible brecha de sobreajuste. Una brecha pequeña no prueba capacidad futura.

Para explicar el modelo se calcula importancia por permutación en una submuestra de validación: cuánto disminuye average precision al desordenar cada variable. No mide causalidad y las variables correlacionadas pueden repartirse importancia. No se eliminan variables ni se reentrena después de este análisis.

In [ ]:
idx=np.random.default_rng(SEED).choice(len(X_val),size=min(10_000,len(X_val)),replace=False)
imp=permutation_importance(final,X_val.iloc[idx],y_val.iloc[idx],scoring='average_precision',n_repeats=3,random_state=SEED,n_jobs=1)
importancias=pd.DataFrame({'Variable':FEATURES,'Disminucion_AP':imp.importances_mean,'Desviacion':imp.importances_std}).sort_values('Disminucion_AP',ascending=False)
print(importancias.round(4).to_string(index=False))
importancias.to_csv(OUT/'importancia_variables.csv',index=False)
fig,ax=plt.subplots(figsize=(8,4));d=importancias.sort_values('Disminucion_AP')
ax.barh(d.Variable,d.Disminucion_AP,color='#287c8e');ax.set_xlabel('Disminución de AP al permutar');ax.set_title('Importancia predictiva en validación')
plt.tight_layout();plt.savefig(OUT/'importancias.png',dpi=140);plt.show()

## 6 Evaluar resultados y equidad
La selección ya está cerrada. Se compara el desempeño en la muestra de test con los umbrales fijados en validación; no se cambia de ganador si otro modelo obtiene un resultado ligeramente mayor en test.

| Métrica | Qué significa |
|---|---|
| Accuracy | Fracción total de aciertos; puede ser elevada sin detectar casos positivos |
| Precisión | De las alertas, qué proporción corresponde a bajo puntaje |
| Recall | De todos los casos con bajo puntaje, qué proporción detectamos |
| F1 y F2 | Equilibrio entre precisión y recall; F2 prioriza recall |
| ROC AUC | Capacidad de ordenar positivos por encima de negativos entre distintos umbrales |
| AP | Average precision, resumen de precisión–recall; su referencia sin señal es la prevalencia positiva |

Los valores van de 0 a 1. AP no es aquí un área trapezoidal. Se muestran falsos positivos y falsos negativos para evaluar la carga operativa y los casos que quedarían sin detectar.

In [ ]:
X_test=validar_entradas(test);y_test=objetivo(test)
filas=[];pt={}
for nombre,m in modelos.items():
    pt[nombre]=m.predict_proba(X_test)[:,1]
    filas.append({'Modelo':nombre,'Umbral':umbrales[nombre],**medir(y_test,pt[nombre],umbrales[nombre])})
resultados=pd.DataFrame(filas).set_index('Modelo');resultados.to_csv(OUT/'metricas_test.csv')
print(resultados.round(4).to_string())
p=pt[elegido];pred=(p>=umbral).astype(int);cm=confusion_matrix(y_test,pred,labels=[0,1])
tn,fp,fn,tp=cm.ravel()
fig,ax=plt.subplots(1,3,figsize=(16,4))
ConfusionMatrixDisplay(cm,display_labels=['Sin bajo puntaje','Bajo puntaje']).plot(ax=ax[0],colorbar=False,cmap='Blues',values_format='d')
ax[0].set_title('Matriz de confusión');ax[0].set_xlabel('Predicción');ax[0].set_ylabel('Clase real')
RocCurveDisplay.from_predictions(y_test,p,ax=ax[1],name=elegido);ax[1].plot([0,1],[0,1],'--',color='gray');ax[1].set_title('Curva ROC')
PrecisionRecallDisplay.from_predictions(y_test,p,ax=ax[2],name=elegido);ax[2].axhline(y_test.mean(),ls='--',color='gray',label='Prevalencia');ax[2].legend();ax[2].set_title('Curva precisión–recall')
plt.tight_layout();plt.savefig(OUT/'evaluacion_test.png',dpi=140);plt.show()
r=resultados.loc[elegido]
print(f'Modelo seleccionado: {elegido}. Recall {r.Recall:.1%}; precisión {r.Precision:.1%}; F2 {r.F2:.3f}.')
print(f'Detecta {tp:,} casos positivos; omite {fn:,}; genera {fp:,} falsas alertas y {tn:,} negativos correctos.')
print(f'Se revisarían {int(pred.sum()):,} de {len(test):,} registros ({pred.mean():.1%}).')
print('La referencia mayoritaria no identifica positivos aunque su exactitud pueda parecer alta.')

In [ ]:
from IPython.display import Markdown, display
r=resultados.loc[elegido]
resumen_tecnico=(f"## Resultados de la ejecución {RUN_ID}\n\n"
 f"Modelo seleccionado por F2 en validación: **{elegido}**, umbral **{umbral:.2f}**. "
 f"F2 validación: **{val_metricas.loc[elegido,'F2']:.6f}**.\n\n"
 f"En test (n={len(y_test):,}): recall **{r.Recall:.2%}**, precisión **{r.Precision:.2%}**, "
 f"F2 **{r.F2:.6f}**, ROC AUC **{r.ROC_AUC:.6f}**.\n\n"
 f"Detectados: **{tp:,}**; no detectados: **{fn:,}**; falsas alertas: **{fp:,}**; "
 f"negativos correctos: **{tn:,}**. Alertas: **{tp+fp:,}** ({(tp+fp)/len(y_test):.2%}).\n\n"
 "Resultados en datos sintéticos. La comparación próxima entre candidatos no demuestra superioridad estadística. "
 "Las alertas requieren revisión humana; no se ha medido impacto escolar.")
display(Markdown(resumen_tecnico))
(OUT/'resumen_resultados.md').write_text(resumen_tecnico,encoding='utf-8')


### Auditoría de equidad correspondiente a RF-03
Se conserva género, etnia y nivel socioeconómico únicamente en una tabla de auditoría. Se calculan tamaños, casos positivos, recall, precisión y tasa de falsos positivos por subgrupo. Las métricas sin denominador se expresan como NaN, no como cero.

Se incorpora un intervalo de Wilson del 95 % para recall para mostrar incertidumbre de muestreo. Las brechas máximas son descriptivas; no constituyen prueba de discriminación ni certificación de equidad. No se eligen umbrales diferentes por atributo protegido. Esta auditoría sobre datos sintéticos debe repetirse con datos institucionales autorizados antes de cualquier uso real.

In [ ]:
def wilson(aciertos,n,z=1.96):
    if n==0:return (np.nan,np.nan)
    p=aciertos/n;den=1+z*z/n;centro=(p+z*z/(2*n))/den
    margen=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return centro-margen,centro+margen
audit=test[AUDITORIA].copy();audit['real']=y_test.to_numpy();audit['pred']=pred
filas=[]
for atributo in AUDITORIA:
    for grupo,g in audit.groupby(atributo,dropna=False):
        tn_g,fp_g,fn_g,tp_g=confusion_matrix(g.real,g.pred,labels=[0,1]).ravel()
        pos=tp_g+fn_g;neg=tn_g+fp_g;lo,hi=wilson(tp_g,pos)
        filas.append({'Atributo':atributo,'Grupo':str(grupo),'N':len(g),'Positivos':int(pos),
          'Recall':tp_g/pos if pos else np.nan,'Recall_IC95_inf':lo,'Recall_IC95_sup':hi,
          'FPR':fp_g/neg if neg else np.nan,'Precision':tp_g/(tp_g+fp_g) if tp_g+fp_g else np.nan,
          'Tasa_alertas':g.pred.mean()})
equidad=pd.DataFrame(filas);equidad.to_csv(OUT/'auditoria_equidad_restringida.csv',index=False)
print(equidad.round(4).to_string(index=False))
brechas=equidad.groupby('Atributo')[['Recall','FPR']].agg(lambda s:s.max()-s.min())
print('\nBrechas max–min por atributo (proporciones):\n',brechas.round(4).to_string())
print('El nombre restringida indica el destino previsto, no aplica permisos ni cifrado por sí solo.')

### ¿La preparación fue suficiente para el proyecto?
Sí permitió ejecutar un prototipo coherente con matemáticas, controlar las particiones y medir errores y diferencias entre subgrupos. No es suficiente para declarar cumplidos los hitos H4–H7: faltan aprobación de UTP y comité, definición del corte, disponibilidad temporal de entradas, validación institucional, seguridad, dashboard y piloto.

Tampoco se debe trasladar el porcentaje de aciertos a una promesa de beneficio pedagógico. La adopción de al menos 70 % de docentes del primer semestre que aparece en la evaluación 2 es un indicador de uso del piloto, no una métrica de precisión del modelo.

## 7 Utilizar el modelo con cinco nuevos datos
Los cinco perfiles son ficticios, construidos para demostrar inferencia. No se extraen de train ni test y no tienen puntaje real asignado: no permiten calcular exactitud. Se entregan probabilidades estimadas sin calibración adicional.

Se definen dos niveles operativos provisionales: **priorizar revisión** si supera el umbral seleccionado, y **seguimiento habitual** en otro caso. No se inventa un tercer nivel sin acordar su utilidad y sus límites. Ambos requieren interpretación docente; una ausencia de alerta no garantiza ausencia de dificultades.

In [ ]:
nuevos=pd.DataFrame([
 [10,0.73,0.3,0,0,1,0,4,4],
 [11,0.82,0.6,1,0,1,0,3,3],
 [12,0.90,1.0,1,1,0,1,3,2],
 [10,0.97,1.7,1,1,0,1,2,1],
 [9,0.78,0.4,1,0,0,0,4,4]],columns=FEATURES)
nuevos.insert(0,'ID_Caso',[f'DEMO_{i}' for i in range(1,6)])
prob_nuevos=final.predict_proba(validar_entradas(nuevos))[:,1]
cinco=pd.DataFrame({'ID_Caso':nuevos.ID_Caso,'Probabilidad_bajo_rendimiento':prob_nuevos,
 'Alerta':(prob_nuevos>=umbral).astype(int),
 'Nivel':np.where(prob_nuevos>=umbral,'Priorizar revisión','Seguimiento habitual')})
print(cinco.round(4).to_string(index=False))
nuevos.to_csv(OUT/'cinco_casos_nuevos.csv',index=False);cinco.to_csv(OUT/'cinco_predicciones.csv',index=False)

## 8 Proponer e implementar una automatización
**Propuesta alineada con la evaluación 2:** actualizar predicciones por lotes al menos semestralmente. Un lote autorizado produce un CSV para el dashboard docente, una bitácora y un resumen técnico. Se evita enviar mensajes o tomar decisiones académicas automáticamente.

**Implementación de este cuaderno:** guarda el pipeline completo y su configuración; vuelve a cargarlo; verifica entradas; genera predicciones; registra tamaño, duración, faltantes imputados, tasa de alertas, huella de entrada y versión. Cada salida usa un nombre único para no sobreescribir ejecuciones.

RF-04 (historial individual) y RF-05 (alerta por aumento de riesgo entre periodos) requieren identificadores estables y mediciones temporales reales, que el dataset no incluye. Se registra historial de ejecuciones; no se finge disponer de historia longitudinal. La programación semestral en un servicio persistente, accesos por rol, cifrado y auditoría de accesos quedan pendientes. Colab no garantiza ejecución programada con la sesión cerrada.

In [ ]:
# El módulo generado contiene todas las funciones necesarias para inferencia.
MODULO = '"""Inferencia independiente. Cargar únicamente modelos joblib de confianza."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom uuid import uuid4\nimport argparse, hashlib, json, platform\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport joblib\n\ndef validar(datos, modelo):\n    features = modelo[\'features\']\n    faltan = set(features) - set(datos.columns)\n    if faltan:\n        raise ValueError(f\'Faltan columnas: {sorted(faltan)}\')\n    if \'ID_Caso\' not in datos:\n        raise ValueError(\'Falta ID_Caso\')\n    ids = datos[\'ID_Caso\'].astype(\'string\').str.strip()\n    if ids.isna().any() or ids.eq(\'\').any() or ids.duplicated().any():\n        raise ValueError(\'ID_Caso debe ser único, no vacío y no nulo por lote\')\n    x = datos[features].apply(pd.to_numeric, errors=\'raise\')\n    if x.empty or x.isna().all(axis=1).any() or np.isinf(x.to_numpy()).any():\n        raise ValueError(\'Lote vacío, fila sin predictores o valores infinitos\')\n    for c, (lo, hi) in modelo[\'limites\'].items():\n        if not x[c].dropna().between(lo, hi).all():\n            raise ValueError(f\'{c} fuera del dominio del prototipo [{lo}, {hi}]. Revisar; no truncar automáticamente.\')\n    for c in modelo[\'binarias\']:\n        if not x[c].dropna().isin([0, 1]).all():\n            raise ValueError(f\'{c} debe ser 0 o 1\')\n    for c in [\'Grade\', \'FreeTime\', \'GoOut\']:\n        if not (x[c].dropna() % 1 == 0).all():\n            raise ValueError(f\'{c} debe ser entero\')\n    return x, ids\n\ndef procesar_lote(ruta_csv, ruta_modelo, destino, origen=\'demostracion_sintetica\'):\n    m = joblib.load(ruta_modelo)\n    actuales = {\'python\':platform.python_version(), \'numpy\':np.__version__,\n                \'pandas\':pd.__version__, \'sklearn\':sklearn.__version__, \'joblib\':joblib.__version__}\n    for clave, version in m[\'versiones\'].items():\n        if clave in actuales and actuales[clave] != version:\n            raise ValueError(f\'Entorno incompatible: {clave} requiere {version}, instalado {actuales[clave]}\')\n    datos = pd.read_csv(ruta_csv, dtype={\'ID_Caso\':\'string\'})\n    x, ids = validar(datos, m)\n    prob = m[\'pipeline\'].predict_proba(x)[:, 1]\n    if not np.isfinite(prob).all() or not ((prob >= 0) & (prob <= 1)).all():\n        raise ValueError(\'Probabilidades inválidas\')\n    alerta = (prob >= m[\'umbral\']).astype(int)\n    fecha = datetime.now(timezone.utc).isoformat()\n    lote = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S\') + \'_\' + uuid4().hex[:8]\n    salida = pd.DataFrame({\'ID_Caso\':ids, \'Grade\':x[\'Grade\'],\n        \'Probabilidad_bajo_rendimiento\':prob, \'Alerta\':alerta,\n        \'Nivel\':np.where(alerta == 1, \'Priorizar revisión\', \'Seguimiento habitual\'),\n        \'Fecha_UTC\':fecha, \'ID_Lote\':lote, \'Version_modelo\':m[\'version\'],\n        \'Modelo\':m[\'nombre_modelo\'], \'Umbral\':m[\'umbral\'],\n        \'Corte_Math\':m[\'corte_math\'], \'Origen\':origen})\n    destino = Path(destino); destino.mkdir(parents=True, exist_ok=True)\n    archivo = destino / f\'predicciones_{lote}.csv\'\n    salida.to_csv(archivo, index=False, encoding=\'utf-8-sig\')\n    resumen = {\'ID_Lote\':lote, \'Fecha_UTC\':fecha, \'Modelo\':m[\'nombre_modelo\'],\n        \'Version_modelo\':m[\'version\'], \'Umbral\':m[\'umbral\'], \'Filas\':len(salida),\n        \'Alertas\':int(alerta.sum()), \'Tasa_alertas\':float(alerta.mean()),\n        \'Valores_imputados\':int(x.isna().sum().sum()), \'Origen\':origen,\n        \'SHA256_entrada\':hashlib.sha256(Path(ruta_csv).read_bytes()).hexdigest(),\n        \'Archivo\':archivo.name}\n    (destino/f\'resumen_{lote}.json\').write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding=\'utf-8\')\n    historial = destino/\'historial_ejecuciones.csv\'\n    pd.DataFrame([resumen]).to_csv(historial, mode=\'a\', header=not historial.exists(), index=False)\n    # Power BI lee este único archivo; el histórico queda separado.\n    tablero = destino/\'dashboard\'; tablero.mkdir(exist_ok=True)\n    temporal = tablero/\'ultimo_lote.tmp\'\n    salida.to_csv(temporal, index=False, encoding=\'utf-8-sig\')\n    temporal.replace(tablero/\'predicciones_actuales.csv\')\n    return salida, resumen\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--entrada\', required=True)\n    parser.add_argument(\'--modelo\', required=True)\n    parser.add_argument(\'--salida\', required=True)\n    parser.add_argument(\'--origen\', default=\'demostracion_sintetica\')\n    args = parser.parse_args()\n    _, resumen = procesar_lote(args.entrada, args.modelo, args.salida, args.origen)\n    print(json.dumps(resumen, indent=2, ensure_ascii=False))\n'
(OUT/'predecir.py').write_text(MODULO,encoding='utf-8')
paquete={'pipeline':final,'features':FEATURES,'nombre_modelo':elegido,'umbral':umbral,
 'corte_math':CORTE_MATH,'version':'matematicas_v3_'+RUN_ID,'dataset':HANDLE,
 'sklearn':sklearn.__version__,'versiones':versiones,'limites':LIMITES,'binarias':BINARIAS}
RUTA_MODELO=OUT/('modelo_'+RUN_ID+'.joblib')
joblib.dump(paquete,RUTA_MODELO)
import importlib.util
spec=importlib.util.spec_from_file_location('inferencia_matematicas',OUT/'predecir.py')
inferencia=importlib.util.module_from_spec(spec);spec.loader.exec_module(inferencia)
salida_lote,resumen_lote=inferencia.procesar_lote(OUT/'cinco_casos_nuevos.csv',RUTA_MODELO,OUT/'lotes')
assert np.allclose(salida_lote.Probabilidad_bajo_rendimiento,cinco.Probabilidad_bajo_rendimiento)
assert list(salida_lote.Alerta)==list(cinco.Alerta)
invalido=nuevos.copy();invalido.loc[0,'AttendanceRate']=90
try:
    inferencia.validar(invalido,paquete)
except ValueError as error:
    print('Control de escala verificado:',error)
else:
    raise AssertionError('Debía rechazar asistencia 90')
print(salida_lote.to_string(index=False))
print(json.dumps(resumen_lote,ensure_ascii=False,indent=2))


### Trazabilidad y ficha del modelo
Se exporta una ficha con objetivo, configuración, procedencia, métricas, limitaciones y alcance. Los archivos permiten versionar posteriormente el trabajo en Git, pero no se crea un repositorio ni se afirma que se haya hecho un commit.

Para Power BI: importar `predicciones_powerbi_*.csv`, establecer la probabilidad como decimal y usar ID_Caso, Nivel y Fecha_UTC. La tabla de auditoría por atributos protegidos se reserva al comité/DPO; no debe mezclarse con el listado docente. Aquí se separan archivos, pero un entorno real debe hacer cumplir los permisos.

El modelo y el módulo `predecir.py` se conservan juntos con `requirements.txt`. El módulo es independiente del kernel del notebook y valida el entorno, identificadores y dominio de entradas. La carpeta del dashboard contiene solo la última salida; el historial permanece fuera de esa carpeta. No importar todos los lotes a una sola tabla de casos actuales.


In [ ]:
ficha={'nombre':'Clasificación de bajo puntaje en matemáticas','version':paquete['version'],
 'datos_sinteticos':True,'dataset':HANDLE,'corte_provisional':CORTE_MATH,'features':FEATURES,
 'atributos_solo_auditoria':AUDITORIA,'modelo':elegido,'umbral_validacion':umbral,
 'muestreo':TAMANOS,'semilla':SEED,'calidad_archivos':calidad,'versiones_software':versiones,
 'metricas_test':resultados.loc[elegido].to_dict(),
 'limitaciones':['Corte no aprobado por UTP','Datos sin secuencia temporal ni ID de estudiante',
 'Test explorado previamente','Métricas calculadas en muestras','Probabilidades sin calibración adicional',
 'No acredita desempeño ni equidad en un colegio real','Seguridad y programación de producción pendientes']}
(OUT/'ficha_modelo.json').write_text(json.dumps(ficha,indent=2,ensure_ascii=False),encoding='utf-8')
(OUT/'versiones_entorno.json').write_text(json.dumps(versiones,indent=2),encoding='utf-8')
print('Archivos generados:')
for archivo in sorted(OUT.rglob('*')):
    if archivo.is_file():print(archivo.relative_to(OUT))

### Seguimiento del proyecto y controles pendientes
| Actividad de la evaluación 2 | Evidencia de esta actividad | Estado |
|---|---|---|
| 02 Acordar objetivo métricas y umbrales | Objetivo matemáticas, F2 y corte provisional documentados | Propuesta; falta aprobación |
| 08 a 11 Calidad preparación y separación | Controles completos de archivos y pipeline reproducible | Prototipo ejecutado |
| 12 y 13 Entrenamiento y ajuste | Referencia, dos candidatos y búsqueda en validación | Ejecutado |
| 15 Evaluación por subgrupo | Métricas y exportación de equidad | Ejecutado sobre datos sintéticos |
| 16 H4 Revisión del comité | Ficha y resultados disponibles | Pendiente de aprobación |
| 18 y 19 Dashboard e integración por lotes | Exportación para Power BI y función de inferencia | Parcial; sin dashboard ni accesos por rol |
| 20 H5 Seguridad y recuperación | No se simulan aprobaciones ni cifrado institucional | Pendiente |
| 24 a 26 Piloto aceptación y operación | No realizado por este notebook | Pendiente |

El equipo informó que ya asignó tareas en Trello y marcó una primera como realizada. Completar enlaces, fechas reales y evidencia de aceptación; el notebook no modifica ni verifica automáticamente el tablero. El calendario de 26 semanas y el presupuesto de las evaluaciones se mantienen como plan, sin atribuir costos ficticios a esta ejecución.

**Monitoreo propuesto, coherente con el plan:** revisión semanal de lotes e incidencias por DE; revisión mensual de desempeño cuando existan etiquetas, deriva y equidad por DS/JP; revisión trimestral de riesgos por JP/DPO; actualización al menos semestral y auditoría antes de desplegar o reentrenar. La tasa de alertas o los faltantes pueden señalar un cambio de datos, pero por sí solos no demuestran deterioro predictivo.

## Reflexión grupal
**¿Qué componente tendrá mayor impacto en el éxito del negocio: el modelo, la automatización o la forma de comunicar los resultados? ¿Por qué?**

Como grupo, consideramos que la comunicación de los resultados tendría el mayor impacto. El proyecto busca que los docentes puedan identificar a quién conviene ofrecer apoyo en matemáticas. Un modelo con buenas métricas no cumpliría ese propósito si el docente no entiende una alerta o la interpreta como una etiqueta definitiva del estudiante.

Por eso, explicaríamos el nivel de revisión, la incertidumbre y los errores del modelo con un lenguaje sencillo. La automatización permitiría actualizar la información sin repetir todo manualmente y el modelo ayudaría a priorizar los casos. Sin embargo, la decisión debe quedar en manos del equipo docente, considerando el contexto del estudiante. Esta elección también se relaciona con la meta del proyecto: lograr que al menos el 70 % de los docentes use regularmente el dashboard durante el primer semestre, indicador que solo podrá comprobarse en el piloto.

## Conclusión
Se implementó el flujo técnico del proyecto con el objetivo correcto de matemáticas, usando muestras independientes de los tres archivos originales. Se documentó un criterio provisional de bajo rendimiento, se separó el ajuste de la evaluación y se incorporó una auditoría de equidad y una automatización demostrable.

Las métricas obtenidas describen este experimento sintético y el corte elegido. La preparación permite continuar con la revisión del docente y los responsables del proyecto; no permite declarar listo un sistema de alerta temprana institucional. El siguiente paso es acordar el corte y las entradas disponibles antes de la predicción, y luego validar el modelo con datos institucionales autorizados y temporalmente separados.

## 9 Preparar una demostración para el dashboard
Se reutilizan 1.000 registros de la muestra de test ya evaluada, seleccionados con semilla fija, **solo para visualizar el prototipo**. No son nuevos estudiantes ni una validación externa. Se excluyen puntajes y atributos de auditoría del CSV docente; el ID identifica la fila sintética, no una persona.

La selección no depende del puntaje ni de las predicciones. No se reentrena ni se elige el modelo con este lote. No debe presentarse su tasa de alertas como la del colegio de 1.500 estudiantes. Los cinco casos manuales siguen disponibles como prueba independiente de inferencia.


In [ ]:
demo=test.sample(n=min(1000,len(test)),random_state=SEED+100).copy()
lote_demo=demo[FEATURES].copy()
lote_demo.insert(0,'ID_Caso','SINT_TEST_'+demo['_fila_fuente'].astype(str))
lote_demo.to_csv(OUT/'entrada_dashboard_demo.csv',index=False)
salida_demo,resumen_demo=inferencia.procesar_lote(
    OUT/'entrada_dashboard_demo.csv',RUTA_MODELO,OUT/'lotes',origen='demo_submuestra_test_sintetico')
assert salida_demo.ID_Caso.is_unique
assert len(salida_demo)==len(lote_demo)
assert not set(AUDITORIA+['TestScore_Math','GPA']) & set(salida_demo.columns)
assert salida_demo.Alerta.eq(salida_demo.Probabilidad_bajo_rendimiento.ge(umbral).astype(int)).all()
print('CSV para Power BI:',(OUT/'lotes/dashboard/predicciones_actuales.csv').resolve())
print('Casos:',len(salida_demo),'Alertas:',int(salida_demo.Alerta.sum()))
display(salida_demo.head())
# Tabla técnica separada: métricas del test completo de esta ejecución.
metricas_export=resultados.reset_index()
metricas_export['Seleccionado']=metricas_export.Modelo.eq(elegido)
metricas_export['ID_Ejecucion']=RUN_ID
metricas_export['N_Test']=len(y_test)
metricas_export.to_csv(OUT/'metricas_dashboard_tecnico.csv',index=False,encoding='utf-8-sig')
pd.DataFrame([{'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'Modelo':elegido,
               'ID_Ejecucion':RUN_ID}]).to_csv(OUT/'confusion_modelo_seleccionado.csv',index=False)


### Diccionario y diseño de Power BI
| Campo | Tipo y uso |
|---|---|
| ID_Caso | Texto; identificador único dentro del lote |
| Grade | Entero; nivel 9–12 del dataset. No traducir automáticamente a cursos chilenos |
| Probabilidad_bajo_rendimiento | Decimal 0–1; formato porcentaje; no sumar |
| Alerta | Entero 0/1; su suma cuenta alertas |
| Nivel | Categoría para filtrar |
| ID_Lote y Version_modelo | Texto para trazabilidad |
| Fecha_UTC | Fecha de procesamiento, no fecha académica ni evolución escolar |
| Modelo, Umbral y Corte_Math | Configuración que produjo la señal |
| Origen | Identifica la demostración sintética |

**Montaje:** importar el único archivo `lotes/dashboard/predicciones_actuales.csv` como tabla **Predicciones**; verificar tipos, usando configuración regional que interprete el punto decimal si hace falta. Crear tarjetas de casos, alertas y porcentaje; barras por Grade; tabla de ID_Caso, nivel y probabilidad; filtros Grade y Nivel. Mostrar siempre “Demostración con datos sintéticos”.

Medidas DAX (adaptar separadores si su entorno lo requiere):
```dax
Casos = COUNTROWS(Predicciones)
Alertas = SUM(Predicciones[Alerta])
Porcentaje alertas = DIVIDE([Alertas], [Casos], 0)
```
No denominar “precisión” al porcentaje de alertas. Para la página técnica, importar `metricas_dashboard_tecnico.csv` y filtrar Seleccionado = verdadero; mantener sus métricas separadas de los filtros de los 1.000 casos. El test evaluado tiene 150.000 registros, mientras que el lote visual es una submuestra.

Guardar el `.pbix`, una captura y el enlace/evidencia en Trello. Comprobar que los conteos del dashboard coincidan con la salida de la celda anterior. Para actualizar, ejecutar inferencia y después actualizar Power BI; este cuaderno no configura actualización programada del servicio.


## 10 Paquete de entrega y predicción sin reentrenar
La siguiente celda exporta las versiones instaladas realmente, las instrucciones de inferencia y una plantilla de trazabilidad. El ZIP reúne los resultados de esta ejecución para que no se pierdan al cerrar el entorno. Guardar también este notebook ejecutado; el ZIP no lo incluye automáticamente.


In [ ]:
import importlib.metadata, shutil
paquetes=['numpy','pandas','scipy','scikit-learn','joblib','threadpoolctl','matplotlib']
if importlib.util.find_spec('kagglehub') is not None: paquetes.append('kagglehub')
requisitos='\n'.join(f'{nombre}=={importlib.metadata.version(nombre)}' for nombre in paquetes)+'\n'
(OUT/'requirements.txt').write_text(requisitos,encoding='utf-8')
comando=f'python predecir.py --entrada entrada_dashboard_demo.csv --modelo {RUTA_MODELO.name} --salida lotes --origen demo_submuestra_test_sintetico'
instrucciones=f"""# Uso del paquete
Python de entrenamiento: {platform.python_version()}
Desde esta carpeta, crear/activar un entorno con esa versión de Python.
Instalar: python -m pip install -r requirements.txt
Cargar solamente archivos joblib propios o de confianza.
Ejecutar: {comando}
El CSV actualizado queda en lotes/dashboard/predicciones_actuales.csv.
No necesita Kaggle ni reentrenamiento para esta predicción.
No mezclar este modelo con resultados de otra carpeta de ejecución.
Los CSV de demostración contienen datos sintéticos; no son evidencia de eficacia escolar.
"""
(OUT/'LEEME_INFERENCIA.md').write_text(instrucciones,encoding='utf-8')
trazabilidad=pd.DataFrame([
 {'Tarea':t,'Evidencia':e,'Responsable':'POR COMPLETAR','Estado':'POR VERIFICAR',
  'Fin_planificado':'','Fin_real':'','Enlace_Trello':'','Revision':''}
 for t,e in [('Preparar datos','calidad_y_procedencia.csv'),
 ('Seleccionar modelo','metricas_validacion.csv'),('Evaluar modelo','metricas_test.csv'),
 ('Auditar subgrupos','auditoria_equidad_restringida.csv'),
 ('Probar inferencia','lotes/historial_ejecuciones.csv'),
 ('Construir dashboard','PBIX y captura pendientes'),('Consolidar informe','Informe EP5 pendiente')]])
trazabilidad.to_csv(OUT/'trazabilidad_para_completar.csv',index=False,encoding='utf-8-sig')
# Los artefactos se identifican por hash para la evidencia de esta versión.
manifest={str(f.relative_to(OUT)):sha256_archivo(f) for f in OUT.rglob('*') if f.is_file() and f.suffix!='.pyc'}
(OUT/'manifest_sha256.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
zip_path=shutil.make_archive(str(OUT),'zip',root_dir=OUT)
print('Carpeta de trabajo:',OUT.resolve())
print('Paquete para descargar:',Path(zip_path).resolve())
print('Modelo elegido:',elegido,'| umbral:',umbral)
print('Comando sin reentrenar:',comando)


## 11 Orden de avance hasta la entrega EP5
1. **Cerrar la revisión técnica.** Reiniciar y ejecutar todo; revisar resumen automático, controles e inferencia. Guardar notebook, ZIP y commit. No copiar cifras antiguas del informe 3.3.2.
2. **Registrar la ejecución.** Adjuntar evidencia a las tarjetas correspondientes. Completar responsable, fechas y revisión; ejecutar código no implica por sí solo aceptación de todas las tareas.
3. **Construir el dashboard.** Utilizar el CSV y las instrucciones del paso 9. Verificar filtros, conteos y aviso de datos sintéticos. Guardar PBIX y captura.
4. **Probar la integración.** Ejecutar `predecir.py` en un proceso independiente, actualizar el dashboard y comprobar que solo muestra el lote actual. Registrar resultado y cualquier incidencia.
5. **Actualizar gestión.** Comparar Gantt base y fechas reales/pronosticadas. Registrar causas, consecuencias, responsables y seguimiento de acciones correctivas. Mantener separados los ejemplos simulados de 3.2.2.
6. **Redactar el informe de 5–8 páginas.** Problema, implementación, resultados de esta ejecución, gestión, desviaciones y conclusiones; incluir enlaces a repositorio, tablero y notebook. Las limitaciones institucionales siguen pendientes aunque el prototipo funcione.
7. **Preparar la presentación de 10 minutos.** Mostrar una predicción y el dashboard, Trello, estado, desviaciones y defensa del plan. No descargar ni entrenar en vivo; llevar capturas de respaldo.
8. **Revisión final grupal.** Confirmar que modelo, umbral y métricas coincidan en notebook, dashboard, informe y presentación. Verificar accesibilidad de enlaces y evidencias individuales.

Distribución propuesta: Genesis revisa datos/inferencia; Diego modelos/métricas; Constanza dashboard; Jimena auditoría/informe; Alejandra seguimiento y consolidación. Ajustar según las asignaciones reales de Trello. La infografía es opcional en la rúbrica recibida.

### Registro de cambios de esta versión
Resumen dinámico; corrección del estado de Trello; exportación docente con Grade y metadatos; lote visual sintético reproducible; módulo independiente de inferencia; carpeta única por ejecución y CSV de lote actual; dependencias, manifiesto y paquete descargable. Se conserva la selección por validación y no se amplía el dominio del modelo sin validación.
